In [5]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')
import distance_utils as dist_util
import monge_rotate_lr as mr_lr

x0 = np.loadtxt('planted_gaussians_k250_sigma0.1_perturb0.1_n2500_X.txt')
x1 = np.loadtxt('planted_gaussians_k250_sigma0.1_perturb0.1_n2500_Y.txt')


In [ ]:
import importlib
importlib.reload(mr_lr)
import jax.numpy as jnp

r = 250
Q, g, R = mr_lr.monge_rotation_kmeans_LR(x0, x1, r, lambda_factor=0.5, random_state=0, epsilon=1e-2)
A, B = dist_util.compute_lr_sqeuclidean_factors(x0, x1, rescale_cost=False)
C = A @ B.T
P = Q @ jnp.diag(1/g) @ R.T

print(f'LR Monge cost is: {jnp.sum(C*P)}')


In [ ]:
jnp.sum(R)

In [ ]:
import FRLC.FRLC as frlc
import torch

def to_torch(x, device=None, dtype=None):
    """
    Robustly convert x (torch / numpy / jax) to a torch.Tensor on (device, dtype).
    Does a defensive copy to avoid dtype/device mismatches.
    """
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)

    # Handle JAX arrays or anything array-like by first making a NumPy array
    try:
        import numpy as np
        x_np = np.asarray(x)
    except Exception:
        # Last resort: construct from list
        x_np = np.array(x)
    return torch.tensor(x_np, device=device, dtype=dtype)  # force dtype/device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
C_t = to_torch(C, device=device, dtype=dtype)   

P, errs = frlc.FRLC_opt(
    C_t, device=device, r=r,
    max_iter=20, returnFull=True,
    gamma=70, max_inneriters_balanced=50,
    max_inneriters_relaxed=50, printCost=True)


In [ ]:

print(f'OT cost of FRLC with random init: {(P * C_t).sum().item()}')
